# End-to-End Evaluation Pipeline

**Duration:** 35 minutes  
**Level:** Advanced  
**Prerequisites:** Complete Notebooks 01-04

## Overview

This notebook demonstrates a **complete BASICS-CDSS evaluation workflow** integrating:
- Scenario generation with perturbations
- Calibration metrics
- Coverage-risk analysis
- Harm-aware evaluation
- Multi-model comparison
- Report generation and export

### What You'll Learn:

1. Complete evaluation workflow from archetypes to reports
2. Multi-model comparison framework
3. Configuration management
4. Report generation and artifact export
5. Reproducibility best practices

### Production-Ready Workflow:

This notebook serves as a template for real CDSS evaluations.

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# Import all BASICS-CDSS modules
from basics_cdss.scenario import (
    instantiate_scenarios,
    PerturbationConfig
)
from basics_cdss.metrics.calibration import (
    calibration_summary
)
from basics_cdss.metrics.coverage_risk import (
    selective_prediction_metrics
)
from basics_cdss.metrics.harm import (
    compute_harm_metrics
)
from basics_cdss.governance.reporting import (
    export_metrics_table,
    export_calibration_plot,
    export_coverage_risk_plot,
    create_reproducibility_manifest
)

print("✓ BASICS-CDSS complete framework loaded")
print(f"Evaluation started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. Evaluation Configuration

Define evaluation parameters for reproducibility.

In [ ]:
# Evaluation configuration
EVAL_CONFIG = {
    'seed': 42,
    'n_per_archetype': 10,
    'perturbation_type': 'composite',
    'perturbation_config': {
        'p_mask': 0.25,
        'noise_sigma': 0.1
    },
    'calibration_bins': 10,
    'coverage_thresholds': 100,
    'harm_weights': {
        'high': 10.0,
        'medium': 3.0,
        'low': 1.0
    },
    'output_dir': 'evaluation_results'
}

print("Evaluation Configuration:")
print("="*60)
for key, value in EVAL_CONFIG.items():
    if key != 'perturbation_config':
        print(f"  {key:<25}: {value}")
    else:
        print(f"  {key:<25}:")
        for k, v in value.items():
            print(f"    {k:<23}: {v}")

# Create output directory
output_dir = Path(EVAL_CONFIG['output_dir'])
output_dir.mkdir(exist_ok=True)
print(f"\n✓ Output directory created: {output_dir}")

## 3. Load/Generate Archetypes

Create sample archetypes for evaluation.

In [ ]:
# Create sample archetypes
np.random.seed(EVAL_CONFIG['seed'])

archetypes_data = [
    {
        'archetype_id': f'A{i:03d}',
        'triage_tier': np.random.choice(['high', 'medium', 'low'], p=[0.2, 0.3, 0.5]),
        'age': np.random.randint(18, 85),
        'heart_rate': np.random.randint(60, 140),
        'temperature': np.random.uniform(36.0, 39.5),
        'blood_pressure_sys': np.random.randint(90, 180),
        'respiratory_rate': np.random.randint(12, 30),
        'oxygen_saturation': np.random.randint(85, 100),
    }
    for i in range(20)  # 20 archetypes
]

archetypes_df = pd.DataFrame(archetypes_data)

print(f"Loaded {len(archetypes_df)} archetypes")
print(f"\nRisk tier distribution:")
print(archetypes_df['triage_tier'].value_counts())
print(f"\nSample archetypes:")
print(archetypes_df[['archetype_id', 'triage_tier', 'age', 'heart_rate']].head())

## 4. Generate Scenarios

Instantiate scenarios from archetypes with perturbations.

In [ ]:
# Create perturbation config
pert_config = PerturbationConfig(
    p_mask=EVAL_CONFIG['perturbation_config']['p_mask'],
    noise_sigma=EVAL_CONFIG['perturbation_config']['noise_sigma']
)

# Generate scenarios
scenarios = instantiate_scenarios(
    archetypes_df,
    n_per_archetype=EVAL_CONFIG['n_per_archetype'],
    seed=EVAL_CONFIG['seed'],
    perturbation_type=EVAL_CONFIG['perturbation_type'],
    perturbation_config=pert_config
)

print(f"Generated {len(scenarios)} scenarios")
print(f"Scenarios per archetype: {EVAL_CONFIG['n_per_archetype']}")

# Analyze uncertainty profiles
missingness = [s.uncertainty_profile['missingness'] for s in scenarios]
ambiguity = [s.uncertainty_profile['ambiguity'] for s in scenarios]

print(f"\nUncertainty statistics:")
print(f"  Missingness: μ={np.mean(missingness):.3f}, σ={np.std(missingness):.3f}")
print(f"  Ambiguity:   μ={np.mean(ambiguity):.3f}, σ={np.std(ambiguity):.3f}")

## 5. Simulate CDSS Models

Create mock predictions from multiple CDSS models for comparison.

In [ ]:
# Extract ground truth from scenarios
risk_tiers = np.array([s.targets['triage_tier'] for s in scenarios])

# Generate ground truth labels
np.random.seed(EVAL_CONFIG['seed'])
n_scenarios = len(scenarios)
y_true = np.zeros(n_scenarios, dtype=int)
y_true[risk_tiers == 'high'] = np.random.binomial(1, 0.75, (risk_tiers == 'high').sum())
y_true[risk_tiers == 'medium'] = np.random.binomial(1, 0.40, (risk_tiers == 'medium').sum())
y_true[risk_tiers == 'low'] = np.random.binomial(1, 0.15, (risk_tiers == 'low').sum())

# Model A: Well-calibrated with good selective prediction
base_probs_a = y_true * 0.75 + (1 - y_true) * 0.25
y_prob_model_a = np.clip(base_probs_a + np.random.normal(0, 0.12, n_scenarios), 0.01, 0.99)
y_pred_model_a = (y_prob_model_a > 0.5).astype(int)

# Model B: Overconfident with poor calibration
base_probs_b = y_true * 0.70 + (1 - y_true) * 0.30
y_prob_model_b = np.where(
    base_probs_b > 0.5,
    base_probs_b + 0.20,
    base_probs_b - 0.15
)
y_prob_model_b = np.clip(y_prob_model_b, 0.01, 0.99)
y_pred_model_b = (y_prob_model_b > 0.5).astype(int)

# Model C: Errors concentrated in high-risk tier (unsafe)
y_pred_model_c = np.copy(y_true)
high_risk_idx = np.where(risk_tiers == 'high')[0]
error_idx = np.random.choice(high_risk_idx, size=int(len(high_risk_idx) * 0.30), replace=False)
y_pred_model_c[error_idx] = 1 - y_pred_model_c[error_idx]
y_prob_model_c = y_pred_model_c.astype(float) + np.random.normal(0, 0.1, n_scenarios)
y_prob_model_c = np.clip(y_prob_model_c, 0.01, 0.99)

print("Model Performance Summary:")
print("="*70)
print(f"{'Model':<20} {'Accuracy':<15} {'Avg Confidence':<20}")
print("-"*70)
print(f"{'Model A (Good)':<20} {(y_pred_model_a == y_true).mean():<15.1%} {y_prob_model_a.mean():<20.3f}")
print(f"{'Model B (Overconfident)':<20} {(y_pred_model_b == y_true).mean():<15.1%} {y_prob_model_b.mean():<20.3f}")
print(f"{'Model C (Unsafe)':<20} {(y_pred_model_c == y_true).mean():<15.1%} {y_prob_model_c.mean():<20.3f}")

print("\n✓ Models simulated successfully")

## 6. Comprehensive Evaluation: Model A

Evaluate Model A across all dimensions.

In [ ]:
print("="*70)
print("COMPREHENSIVE EVALUATION: MODEL A (Good)")
print("="*70)

# 1. Calibration
cal_summary_a = calibration_summary(
    y_true, y_prob_model_a, risk_tiers=risk_tiers,
    n_bins=EVAL_CONFIG['calibration_bins']
)
print("\n1. CALIBRATION METRICS:")
print(f"  ECE:          {cal_summary_a['overall']['ece']:.4f}")
print(f"  Brier Score:  {cal_summary_a['overall']['brier_score']:.4f}")

# 2. Selective Prediction
sp_metrics_a = selective_prediction_metrics(
    y_true, y_prob_model_a,
    n_thresholds=EVAL_CONFIG['coverage_thresholds']
)
print("\n2. SELECTIVE PREDICTION:")
print(f"  AURC:                     {sp_metrics_a.aurc:.4f}")
print(f"  Risk at 80% coverage:     {sp_metrics_a.risk_at_coverage_threshold:.4f}")

# 3. Harm-Aware
harm_metrics_a = compute_harm_metrics(
    y_true, y_pred_model_a, risk_tiers,
    harm_weights=EVAL_CONFIG['harm_weights']
)
print("\n3. HARM-AWARE METRICS:")
print(f"  Weighted Harm Loss:       {harm_metrics_a.weighted_harm_loss:.4f}")
print(f"  Escalation Failures:      {harm_metrics_a.escalation_failures}")
print(f"  Harm Concentration:       {harm_metrics_a.harm_concentration:.2%}")

print("\n✓ Model A evaluation complete")

## 7. Multi-Model Comparison

Compare all three models across all metrics.

In [ ]:
# Evaluate all models
models = {
    'Model A (Good)': (y_prob_model_a, y_pred_model_a),
    'Model B (Overconfident)': (y_prob_model_b, y_pred_model_b),
    'Model C (Unsafe)': (y_prob_model_c, y_pred_model_c)
}

results = []

for model_name, (y_prob, y_pred) in models.items():
    # Calibration
    cal = calibration_summary(y_true, y_prob, risk_tiers, EVAL_CONFIG['calibration_bins'])
    
    # Selective prediction
    sp = selective_prediction_metrics(y_true, y_prob, n_thresholds=EVAL_CONFIG['coverage_thresholds'])
    
    # Harm-aware
    harm = compute_harm_metrics(y_true, y_pred, risk_tiers, EVAL_CONFIG['harm_weights'])
    
    results.append({
        'Model': model_name,
        'Accuracy': (y_pred == y_true).mean(),
        'ECE': cal['overall']['ece'],
        'Brier': cal['overall']['brier_score'],
        'AURC': sp.aurc,
        'Harm Loss': harm.weighted_harm_loss,
        'Esc. Failures': harm.escalation_failures,
        'Harm Conc.': harm.harm_concentration
    })

# Create comparison DataFrame
comparison_df = pd.DataFrame(results)

print("Multi-Model Comparison: Beyond-Accuracy Metrics")
print("="*120)
print(comparison_df.to_string(index=False))

print("\n" + "="*120)
print("KEY INSIGHTS:")
print("-"*120)

# Find best models for each metric
best_ece = comparison_df.loc[comparison_df['ECE'].idxmin()]
best_aurc = comparison_df.loc[comparison_df['AURC'].idxmin()]
best_harm = comparison_df.loc[comparison_df['Harm Loss'].idxmin()]

print(f"Best Calibration:        {best_ece['Model']} (ECE={best_ece['ECE']:.4f})")
print(f"Best Selective Pred:     {best_aurc['Model']} (AURC={best_aurc['AURC']:.4f})")
print(f"Lowest Harm:             {best_harm['Model']} (Harm={best_harm['Harm Loss']:.4f})")

# Identify unsafe models
unsafe = comparison_df[comparison_df['Esc. Failures'] > 5]
if len(unsafe) > 0:
    print(f"\n⚠️  UNSAFE MODELS (>5 escalation failures):")
    for _, row in unsafe.iterrows():
        print(f"  - {row['Model']}: {row['Esc. Failures']} failures")

print("\n✓ Multi-model comparison complete")

## 8. Export Results and Generate Reports

Save evaluation artifacts for documentation and reproducibility.

In [ ]:
print("Exporting Evaluation Artifacts...")
print("="*70)

# 1. Export comparison table
comparison_path = output_dir / "model_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)
print(f"✓ Model comparison saved: {comparison_path}")

# 2. Export calibration metrics
cal_metrics_path = output_dir / "calibration_metrics.csv"
export_metrics_table(cal_summary_a, cal_metrics_path)
print(f"✓ Calibration metrics saved: {cal_metrics_path}")

# 3. Export calibration plot (Model A)
cal_plot_path = output_dir / "calibration_plot.png"
rel_curve = cal_summary_a['overall']['reliability_curve']
export_calibration_plot(
    rel_curve[0], rel_curve[1], cal_plot_path,
    title="Model A: Calibration Reliability"
)
print(f"✓ Calibration plot saved: {cal_plot_path}")

# 4. Export coverage-risk plot (Model A)
cr_plot_path = output_dir / "coverage_risk_plot.png"
export_coverage_risk_plot(
    sp_metrics_a.coverage_curve,
    sp_metrics_a.risk_curve,
    cr_plot_path,
    title="Model A: Coverage-Risk Tradeoff"
)
print(f"✓ Coverage-risk plot saved: {cr_plot_path}")

# 5. Create reproducibility manifest
manifest_path = output_dir / "REPRODUCIBILITY.yaml"
data_info = {
    'n_archetypes': len(archetypes_df),
    'n_scenarios': len(scenarios),
    'risk_tier_distribution': risk_tiers.tolist(),
    'evaluation_config': EVAL_CONFIG
}
result_files = {
    'comparison': comparison_path,
    'calibration_metrics': cal_metrics_path,
    'calibration_plot': cal_plot_path,
    'coverage_risk_plot': cr_plot_path
}

# Note: create_reproducibility_manifest expects a config file path
# For this demo, we'll save config and reference it
import yaml
config_path = output_dir / "config.yaml"
with open(config_path, 'w') as f:
    yaml.dump(EVAL_CONFIG, f)

create_reproducibility_manifest(
    config_path,
    data_info,
    result_files,
    manifest_path
)
print(f"✓ Reproducibility manifest saved: {manifest_path}")

print("\n" + "="*70)
print("EVALUATION COMPLETE")
print("="*70)
print(f"All artifacts saved to: {output_dir.absolute()}")
print(f"Evaluation finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 9. Best Practices and Recommendations

### Production Deployment Checklist:

1. **Calibration** (ECE < 0.10):
   - ✓ Verify calibration on test set
   - ✓ Check stratified calibration by risk tier
   - ✗ Recalibrate if ECE > 0.10

2. **Selective Prediction** (AURC minimized):
   - ✓ Set abstention threshold based on acceptable coverage
   - ✓ Monitor abstention rate in production
   - ✓ Ensure abstention improves conditional accuracy

3. **Harm-Aware** (Escalation failures minimized):
   - ✓ Zero tolerance for escalation failures
   - ✓ Monitor harm concentration (should be <0.5)
   - ✓ Adjust harm weights for your clinical context

4. **Reproducibility**:
   - ✓ Fixed random seeds for all experiments
   - ✓ Version-controlled configuration files
   - ✓ Comprehensive logging and audit trails

### Red Flags (Do Not Deploy):

- **ECE > 0.15**: Severe miscalibration
- **AURC > 0.30**: Poor selective prediction
- **Escalation failures > 0**: Missing critical cases
- **Harm concentration > 0.70**: Errors in high-risk tier

## 10. Summary and Next Steps

### What We Accomplished:

1. ✓ Complete evaluation workflow from archetypes to reports
2. ✓ Multi-model comparison across calibration, selective prediction, and harm
3. ✓ Automated artifact generation (tables, plots, manifests)
4. ✓ Reproducible evaluation pipeline

### Key Evaluation Metrics:

| Dimension | Metric | Threshold |
|-----------|--------|-----------||
| Calibration | ECE | < 0.10 |
| Selective Prediction | AURC | < 0.20 |
| Harm-Aware | Escalation Failures | 0 |
| Harm-Aware | Harm Concentration | < 0.50 |

### Production Workflow:

```python
# 1. Load real archetypes
from basics_cdss.scenario import load_archetypes_csv
archetypes = load_archetypes_csv("data/syndx_archetypes.csv")

# 2. Generate scenarios
scenarios = instantiate_scenarios(archetypes, n_per_archetype=50, seed=42)

# 3. Get CDSS predictions
y_prob = your_cdss_model.predict_proba(scenarios)

# 4. Comprehensive evaluation
cal_metrics = calibration_summary(y_true, y_prob, risk_tiers)
sp_metrics = selective_prediction_metrics(y_true, y_prob)
harm_metrics = compute_harm_metrics(y_true, y_pred, risk_tiers)

# 5. Export results
export_metrics_table(cal_metrics, "results/calibration.csv")
create_reproducibility_manifest(config, data_info, results, "REPRODUCIBILITY.yaml")
```

### Further Reading:

- BASICS-CDSS Manuscript (Section 3: Methodology)
- API Documentation: `docs/api_reference.md`
- Example Evaluations: `examples/`

### Support:

- GitHub Issues: Report bugs or request features
- Documentation: Complete API reference and examples
- Citation: Please cite the BASICS-CDSS manuscript when using this framework

**Congratulations!** You've completed the BASICS-CDSS tutorial series. You're now equipped to evaluate CDSS systems beyond accuracy using behavioral safety metrics.